## IMDB Sentiment Classifier with BERT

A minimal walkthrough of fine-tuning **BERT** on the IMDB movie reviews dataset for binary sentiment classification (positive / negative).

**What we'll do:**
1. Install libraries and set up
2. Load the IMDB dataset
3. Look at the data
4. Split into train, validation and test
5. Tokenize with the BERT tokenizer
6. Load pre-trained BERT
7. Fine-tune with Hugging Face `Trainer`
8. Evaluate on the test set
9. Predict sentiment on new reviews
10. Save the model

## 1. Install libraries

In [ ]:
%pip install -q transformers datasets scikit-learn accelerate

## 2. Imports & Setup

In [2]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, f1_score

In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 3. Load the IMDB Dataset

We load IMDB directly from Hugging Face's `datasets` hub — no Kaggle login needed. It comes pre-split into `train` and `test` (25,000 reviews each).

In [5]:
dataset = load_dataset("imdb")

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## 4. Look at a sample

Each example has two fields:
- `text`: the movie review
- `label`: `0` = negative, `1` = positive

In [7]:
sample = dataset["train"][0]
print("Label:", sample["label"], "(0 = negative, 1 = positive)")
print("\nReview:", sample["text"][:500], "...")

Label: 0 (0 = negative, 1 = positive)

Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent ...


## 5. Take the dataset

In [8]:
trainset = dataset["train"].shuffle(seed=SEED)
testset = dataset["test"].shuffle(seed=SEED)

# split the training data into train + validation
split = trainset.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("Test size :", len(testset))

Train size: 22500
Val size: 2500
Test size : 25000


## 6. Tokenize the text

In [9]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
  return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# tokenize all three sets
train_tokenized = train_ds.map(tokenize, batched=True)
val_tokenized = val_ds.map(tokenize, batched=True)
test_tokenized = testset.map(tokenize, batched=True)

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

## 7. Load pre-trained BERT for Classification

`AutoModelForSequenceClassification` loads BERT and adds a small classification head on top (`num_labels=2` for positive/negative). The BERT body starts pre-trained; the classification head starts random. Fine-tuning trains both together on our data.

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 8. Metrics and Data Collator

- **Metrics**: accuracy and F1 score, computed after each epoch.
- **Data collator**: pads each batch to the length of the longest review *in that batch* (more efficient than padding everything to 256).

In [11]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  return {
      "accuracy": accuracy_score(labels, preds),
      "f1": f1_score(labels, preds)
  }

In [12]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 9. Set up Trainer

In [13]:
training_args = TrainingArguments(
    output_dir="./bert_imdb_output",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 10. Fine-tuning the model

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.318900,0.297669,0.910400,0.912226
2,0.099200,0.358893,0.916800,0.916733


TrainOutput(global_step=5626, training_loss=0.24924369778787372, metrics={'train_runtime': 971.1046, 'train_samples_per_second': 46.339, 'train_steps_per_second': 5.793, 'total_flos': 5914140413508000.0, 'train_loss': 0.24924369778787372, 'epoch': 2.0})

## 11. Evaluate on test set

In [15]:
final_results = trainer.evaluate(eval_dataset=test_tokenized)
print("Final results:")
for k, v in final_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

Final results:
  eval_loss: 0.3442
  eval_accuracy: 0.9202
  eval_f1: 0.9205
  eval_runtime: 151.8849
  eval_samples_per_second: 164.5980
  eval_steps_per_second: 20.5750
  epoch: 2.0000


## 12. Predict sentiment on new reviews

In [16]:
id2label = {0: "negative", 1: "positive"}

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "label": id2label[pred_id],
        "confidence": round(probs[pred_id].item(), 4),
    }

# Try it out
reviews = [
    "This movie was absolutely fantastic. Brilliant acting and a gripping story.",
    "I hated this film. It was boring, too long, and badly written.",
    "It was okay — some good moments, but overall forgettable.",
]

for r in reviews:
    print(predict_sentiment(r), "->", r[:60], "...")

{'label': 'positive', 'confidence': 0.9991} -> This movie was absolutely fantastic. Brilliant acting and a  ...
{'label': 'negative', 'confidence': 0.9989} -> I hated this film. It was boring, too long, and badly writte ...
{'label': 'negative', 'confidence': 0.9955} -> It was okay — some good moments, but overall forgettable. ...


## 13. Saving the Model and Tokenizer

In [17]:
SAVE_DIR = "./imdb_bert"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer to:", SAVE_DIR)

Saved model and tokenizer to: ./imdb_bert


## Wrap-up

That's a full BERT sentiment classifier in ~80 lines of code. What we covered:

- Loaded IMDB from Hugging Face `datasets`
- Tokenized reviews with the BERT tokenizer
- Loaded `bert-base-uncased` with a classification head
- Fine-tuned using `Trainer` (handles loop, eval, mixed precision, checkpointing)
- Evaluated with accuracy and F1
- Ran inference on new text
- Saved the trained model

**Ideas to extend this:**
- Use 3–4 epochs — accuracy should exceed 93%
- Try `distilbert-base-uncased` — ~60% the size of BERT, almost as accurate, much faster
- Plot the training loss curve from `trainer.state.log_history`